# Topic 10 — Combining DataFrames with Pandas

## `concat()` · `merge()` · `join()` — from basics to professional use

> **Analogy:** Think of DataFrames as sheets of paper.
> - **`concat`** = **stacking** sheets on top of each other (more rows) or taping them side by side (more columns).
> - **`merge`** = **matching** two lists by a shared ID, like matching a customer list with an order list by customer number.
> - **`join`** = a shortcut for merge that matches on the **index** (row labels).

### Roadmap

| Part | Sections |
|---|---|
| 🟢 **Why & concat** | 1 Why combine · 2 `concat()` · 3 Row-wise · 4 Column-wise · 5 Different columns |
| 🟡 **merge** | 6 `merge()` · 7 Join types · 8 Visual explanation · 9 Different key names · 10 Multiple keys · 11 Suffixes · 12 Relationship types · 13 `validate` · 14 `indicator` |
| 🟡 **join & comparison** | 15 `join()` · 16 merge vs join vs concat |
| 🔴 **Safety & real world** | 17 Duplicate keys · 18 Preventing bad merges · 19 Three real tables · 20 Advanced (multi-merge, aggregation, MultiIndex, `merge_asof`, `merge_ordered`, many files) · 21 Common mistakes |
| 📝 **Practice** | 20 problems → Solutions |
| 🚀 **Project** | E-Commerce Database Integration |
| 📋 **Reference** | Cheat sheet |

Every major concept: **What · Why · When · When not · Syntax · Small example · Expected output · Explanation · Common mistakes**.

> 📚 **Course navigation** — you are in **Topic 10: Combining DataFrames**
> ⬅️ Previous: Topic 9 — Pandas Data Manipulation  ·  ➡️ Next: Topic 11 — Feature Engineering

<details><summary>Full course sequence (click to expand)</summary>

| # | Topic | Notebook |
|---|---|---|
| 01 | Python Basics | `Topic_01_Python_Basics.ipynb` |
| 02 | Pandas DataFrame Basics | `Topic_02_Pandas_DataFrame_Basics.ipynb` |
| 03 | Handling Missing Values | `Topic_03_Handling_Missing_Values.ipynb` |
| 04 | Normalization & Standardization | `Topic_04_Normalization_Standardization.ipynb` |
| 05 | Detecting & Treating Outliers | `Topic_05_Outliers.ipynb` |
| 06 | Encoding Techniques | `Topic_06_Encoding_Techniques.ipynb` |
| 07 | Exploratory Data Analysis (EDA) | `Topic_07_Exploratory_Data_Analysis.ipynb` |
| 08 | Data Visualization | `Topic_08_Data_Visualization.ipynb` |
| 09 | Pandas Data Manipulation | `Topic_09_Pandas_Data_Manipulation.ipynb` |
| **▶ 10 ◀** | **Combining DataFrames** | `Topic_10_Combining_DataFrames.ipynb` |
| 11 | Feature Engineering | `Topic_11_Feature_Engineering.ipynb` |
| 11b | KBinsDiscretizer (Discretization) | `Topic_11b_KBinsDiscretizer.ipynb` |
| 12 | Correlation & Multicollinearity | `Topic_12_Correlation_Multicollinearity.ipynb` |
| 13 | Data Transformation & Skewness | `Topic_13_Data_Transformation_Skewness.ipynb` |
| 14 | Statistics for Data Science | `Topic_14_Statistics_for_Data_Science.ipynb` |
| 15 | Probability for Data Science & ML | `Topic_15_Probability_for_Data_Science_ML.ipynb` |
| 16 | Feature Selection | `Topic_16_Feature_Selection.ipynb` |
| 17 | Machine Learning | `Topic_17_Machine_Learning.ipynb` |

</details>

## 📋 Lesson Index — What This Lesson Covers

This lesson has **33 sections**. Click any one to jump to it.

- [1. Why Do We Combine DataFrames?](#sec-1)
- [2. `pd.concat()`](#sec-2)
- [3. Row-wise Concatenation (monthly datasets)](#sec-3)
- [4. Column-wise Concatenation — `axis=1`](#sec-4)
- [5. `concat()` with Different Columns](#sec-5)
- [6. `pd.merge()` — joining tables on a key](#sec-6)
- [7. Types of Merge](#sec-7)
- [7.1 Inner join — `how='inner'`](#sec-8)
- [7.2 Left join — `how='left'`](#sec-9)
- [7.3 Right join — `how='right'`](#sec-10)
- [7.4 Outer join — `how='outer'`](#sec-11)
- [8. Visual Explanation of Joins](#sec-12)
- [9. Merge on Different Column Names — `left_on` / `right_on`](#sec-13)
- [10. Multiple Keys (composite keys)](#sec-14)
- [11. Merge with Suffixes](#sec-15)
- [12. Relationship Types: One-to-One, One-to-Many, Many-to-One, Many-to-Many](#sec-16)
- [13. Validate the Merge — `validate=`](#sec-17)
- [14. Merge Indicator — `indicator=True`](#sec-18)
- [15. `df.join()`](#sec-19)
- [16. `merge` vs `join` vs `concat`](#sec-20)
- [17. Handling Duplicate Keys](#sec-21)
- [18. Preventing Bad Merges — Checklist](#sec-22)
- [19. Combining Real-World Tables: Customers + Orders + Products](#sec-23)
- [20. Advanced Combining](#sec-24)
- [20.1 Merge multiple DataFrames](#sec-25)
- [20.2 Merge after aggregation](#sec-26)
- [20.3 Merge with indexes, 20.4 MultiIndex joins](#sec-27)
- [20.5 `pd.merge_asof()` — nearest-key (time) matching](#sec-28)
- [20.6 `pd.merge_ordered()` — ordered outer merge with filling](#sec-29)
- [20.7 Combining time-series data & 20.8 concatenating many files/DataFrames](#sec-30)
- [21. Common Mistakes](#sec-31)
- [Final Project — E-Commerce Database Integration](#sec-32)
- [Final Cheat Sheet](#sec-33)

> 🗂️ Full course list: [Topic_00_All_Topics_Index.ipynb](Topic_00_All_Topics_Index.ipynb)

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 30)
print('pandas', pd.__version__)

pandas 3.0.3


---
<a id="sec-1"></a>
# 1. Why Do We Combine DataFrames?

### Real data lives in several tables
Databases split information into separate tables so each fact is stored **once** (this is called **normalization**):

| Table | One row = | Example columns |
|---|---|---|
| **Customers** | one customer | Customer_ID, Name, City, Age |
| **Orders** | one order | Order_ID, Customer_ID, Product_ID, Quantity, Date |
| **Products** | one product | Product_ID, Product, Category, Price |
| **Employees** | one employee | Emp_ID, Name, Dept_ID |
| **Sales** | one sale | Sale_ID, Emp_ID, Amount |
| **Monthly files** | one month's data | same columns in `jan.csv`, `feb.csv`, `mar.csv` |

### Why not one giant table?
- If a customer moves city, you update **one** row in Customers, not 500 order rows.
- Less storage and fewer inconsistencies (no "Lahore" in one order and "Karachi" in another for the same customer).

### But for analysis we need them together
"Revenue per **city**" needs quantities (Orders) × prices (Products) × city (Customers) → we must **combine** the tables.

```text
Customers ──Customer_ID──▶ Orders ◀──Product_ID── Products
      (merge on keys)               (merge on keys)

jan.csv
feb.csv   ──stack──▶  all months   (concat)
mar.csv
```

---
<a id="sec-2"></a>
# 2. `pd.concat()`

### What
Glues DataFrames together **along an axis** — no matching on keys, just stacking (rows) or placing side by side (columns, aligned by index).

### Syntax
```python
pd.concat(
    [df1, df2, ...],        # a LIST of DataFrames
    axis=0,                 # 0 = stack rows (default) · 1 = side by side (columns)
    ignore_index=False,     # True → new index 0..n-1
    keys=None,              # labels for each piece → MultiIndex (remember where rows came from)
    join='outer',           # 'outer' = keep all columns · 'inner' = only shared columns
)
```

| Parameter | Effect |
|---|---|
| `axis=0` | Rows of df2 go **below** df1 |
| `axis=1` | Columns of df2 go **to the right** of df1 (rows matched by **index**) |
| `ignore_index=True` | Throw away old indexes, create 0, 1, 2 … (avoids duplicate index labels) |
| `keys=['a', 'b']` | Add an outer index level showing the source of each row |
| `join='inner'` | Keep only columns (axis=0) / index labels (axis=1) present in **all** pieces |

### Why / When
Same kind of data split into pieces: monthly files, data from several branches, results of a loop, adding computed columns that share the same index.

### When not
Matching rows by an ID column (customer ↔ order) → that's **merge**.

In [ ]:
df1 = pd.DataFrame({'Name': ['Ali', 'Sara'], 'Score': [85, 92]})
df2 = pd.DataFrame({'Name': ['Omar', 'Hina'], 'Score': [78, 88]})

print('pd.concat([df1, df2]) — note the repeated index 0, 1, 0, 1:')
print(pd.concat([df1, df2]), '\n')
print('ignore_index=True:')
print(pd.concat([df1, df2], ignore_index=True), '\n')
print("keys=['Class A', 'Class B']:")
print(pd.concat([df1, df2], keys=['Class A', 'Class B']))

### Expected output & explanation
- Plain `concat` keeps the original indexes → labels `0, 1, 0, 1` (duplicates! `.loc[0]` would return 2 rows).
- `ignore_index=True` gives a clean `0, 1, 2, 3`.
- `keys=` adds an outer level (`Class A`/`Class B`) so you still know where each row came from.

### Common mistakes
Passing DataFrames without a list (`pd.concat(df1, df2)` → error) · forgetting `ignore_index=True` · calling concat inside a loop (slow — collect pieces in a list and concat **once**, Section 20).

---
<a id="sec-3"></a>
# 3. Row-wise Concatenation (monthly datasets)

Monthly exports usually have **identical columns**. Stack them into one table and remember the month.

In [ ]:
jan = pd.DataFrame({'Order_ID': [1, 2, 3], 'Product': ['Pen', 'Book', 'Bag'], 'Amount': [50, 400, 1200]})
feb = pd.DataFrame({'Order_ID': [4, 5], 'Product': ['Pen', 'Lamp'], 'Amount': [60, 2200]})
mar = pd.DataFrame({'Order_ID': [6, 7, 8], 'Product': ['Book', 'Bag', 'Pen'], 'Amount': [380, 1150, 55]})

q1 = pd.concat([jan, feb, mar], keys=['Jan', 'Feb', 'Mar'], names=['Month', 'Row']).reset_index(level='Month').reset_index(drop=True)
print(q1, '\n')
print('Total per month:'); print(q1.groupby('Month', sort=False)['Amount'].sum())

### Explanation
`keys` + `names` created a `Month` index level; `reset_index(level='Month')` turned it into a normal column, and `reset_index(drop=True)` gave a clean 0..7 index. An alternative is to add the column before concatenating: `jan.assign(Month='Jan')`.

### Common mistakes
Losing track of which file a row came from (use `keys` or add a column) · columns with slightly different names (`Amount` vs `amount`) → they become **separate** columns full of NaN (Section 5).

---
<a id="sec-4"></a>
# 4. Column-wise Concatenation — `axis=1`

### What
Places DataFrames **side by side**. Rows are matched by their **index labels**, not by position!

### When
Adding columns that were computed from the same rows (same index), e.g. features created in separate steps.

### When not
Combining tables that share an ID column but have different row orders/indexes → use **merge**.

In [ ]:
info = pd.DataFrame({'Name': ['Ali', 'Sara', 'Omar']}, index=[0, 1, 2])
scores = pd.DataFrame({'Score': [85, 92, 78]}, index=[0, 1, 2])
print('Same index → rows line up:')
print(pd.concat([info, scores], axis=1), '\n')

scores_shuffled = pd.DataFrame({'Score': [78, 85, 92]}, index=[2, 0, 1])     # same data, different order
print('Different order but same labels → still aligned correctly by index:')
print(pd.concat([info, scores_shuffled], axis=1), '\n')

scores_new_index = pd.DataFrame({'Score': [85, 92, 78]}, index=[10, 11, 12])  # e.g. after filtering/sorting elsewhere
print('Different index labels → NO alignment → NaNs and 6 rows:')
print(pd.concat([info, scores_new_index], axis=1), '\n')
print('Fix: reset both indexes if the rows really are in the same order:')
print(pd.concat([info.reset_index(drop=True), scores_new_index.reset_index(drop=True)], axis=1))

### Explanation
`axis=1` performs an **outer join on the index**. When the labels don't match, pandas cannot guess which rows belong together, so it keeps all labels and fills gaps with NaN. `join='inner'` would keep only labels present in both.

### Common mistakes
Assuming `axis=1` matches by position · forgetting that filtering/sorting changes the index before concatenating.

---
<a id="sec-5"></a>
# 5. `concat()` with Different Columns

When pieces have **different columns**:
- `join='outer'` (default): keep **all** columns; missing values become **NaN**.
- `join='inner'`: keep only the columns present in **every** piece.

In [ ]:
branch_a = pd.DataFrame({'Order_ID': [1, 2], 'Amount': [500, 800], 'Coupon': ['SAVE10', None]})
branch_b = pd.DataFrame({'Order_ID': [3, 4], 'Amount': [300, 900], 'Delivery_Fee': [150, 0]})

print("join='outer' (default):")
print(pd.concat([branch_a, branch_b], ignore_index=True), '\n')
print("join='inner':")
print(pd.concat([branch_a, branch_b], ignore_index=True, join='inner'))

### Explanation
- **Outer:** `Coupon` is NaN for branch B's rows and `Delivery_Fee` is NaN for branch A's rows — because those columns didn't exist there.
- **Inner:** only `Order_ID` and `Amount` (shared by both) survive.

### Common mistakes
A typo or different capitalisation in one file silently creates an extra NaN-filled column → standardise column names **before** concatenating (`df.columns.str.strip().str.lower()`).

---
<a id="sec-6"></a>
# 6. `pd.merge()` — joining tables on a key

### What
Combines two DataFrames by **matching values in one or more key columns** — exactly like SQL `JOIN`.

### Syntax
```python
pd.merge(left, right,
         on='Customer_ID',          # key column with the same name in both
         how='inner',               # 'inner' (default), 'left', 'right', 'outer', 'cross'
         left_on=None, right_on=None,   # when the key names differ
         suffixes=('_x', '_y'),     # for overlapping non-key column names
         validate=None,             # check the relationship ('one_to_one', ...)
         indicator=False)           # add a _merge column showing where each row came from
left.merge(right, on='Customer_ID', how='left')   # same thing, method style
```

### Why / When
Related information stored in different tables (customers ↔ orders ↔ products).

### When not
Stacking tables with the same columns → `concat`. Matching on the **index** → `join` (or `merge(left_index=True, right_index=True)`).

In [ ]:
customers = pd.DataFrame({'Customer_ID': [1, 2, 3, 4], 'Name': ['Ali', 'Sara', 'Omar', 'Hina'],
                          'City': ['Lahore', 'Karachi', 'Lahore', 'Islamabad']})
orders = pd.DataFrame({'Order_ID': [101, 102, 103, 104, 105], 'Customer_ID': [1, 2, 1, 5, 3],
                       'Amount': [500, 1200, 300, 900, 750]})
print('customers:'); print(customers, '\n')
print('orders:'); print(orders, '\n')
print("pd.merge(orders, customers, on='Customer_ID'):")
print(pd.merge(orders, customers, on='Customer_ID'))

### Expected output & explanation
Each order gets the name and city of its customer. Ali (ID 1) placed two orders → his details appear twice. Order 104 (customer 5, who doesn't exist in `customers`) and Hina (ID 4, no orders) are **missing** from the result — because the default `how='inner'` keeps only matches. Section 7 shows the other options.

---
<a id="sec-7"></a>
# 7. Types of Merge

We use the same two small tables for every join type:
- **customers**: IDs 1, 2, 3, 4 (Hina = 4 has **no orders**)
- **orders**: customer IDs 1, 2, 1, **5**, 3 (customer 5 **doesn't exist** in customers)

<a id="sec-8"></a>
## 7.1 Inner join — `how='inner'`
**Keeps:** only keys found in **both** tables.
**Use when:** you only care about complete matches (e.g. orders with a known customer).

In [ ]:
inner = pd.merge(customers, orders, on='Customer_ID', how='inner')
print(inner)
print('\nRows:', len(inner), '| Hina (no orders) dropped, order 104 (unknown customer 5) dropped')

<a id="sec-9"></a>
## 7.2 Left join — `how='left'`
**Keeps:** **all rows of the left** table; right-side columns are NaN where there is no match.
**Use when:** the left table is your "main" list and you want to **enrich** it (e.g. all customers + their orders, including customers who never ordered). The most common join in analysis.

In [ ]:
left = pd.merge(customers, orders, on='Customer_ID', how='left')
print(left)
print('\nRows:', len(left), '| Hina kept with NaN order columns; order 104 dropped (its customer is not in the left table)')

<a id="sec-10"></a>
## 7.3 Right join — `how='right'`
**Keeps:** **all rows of the right** table.
**Use when:** the right table is the main one. (Equivalent to a left join with the tables swapped — most people swap and use `left` for readability.)

In [ ]:
right = pd.merge(customers, orders, on='Customer_ID', how='right')
print(right)
print('\nRows:', len(right), '| every order kept; order 104 has NaN Name/City (unknown customer 5); Hina dropped')

<a id="sec-11"></a>
## 7.4 Outer join — `how='outer'`
**Keeps:** **all keys from both** tables; NaN wherever a side has no match.
**Use when:** you need the complete picture, e.g. **reconciling** two systems and finding what's missing on either side.

In [ ]:
outer = pd.merge(customers, orders, on='Customer_ID', how='outer')
print(outer)
print('\nRows:', len(outer), '| Hina (no orders) AND order 104 (unknown customer) are both kept')

### Common mistakes
- Using `inner` by default and silently **losing rows** (always compare row counts before/after).
- Using `left` with the tables in the wrong order.
- Integer columns become **float** after a left/outer join when NaNs appear (e.g. `Order_ID` 101.0) — convert back with `astype('Int64')` if needed.

---
<a id="sec-12"></a>
# 8. Visual Explanation of Joins

```text
          customers (left)            orders (right)
          Customer_ID                 Customer_ID
              1  ◀───── match ─────▶   1, 1
              2  ◀───── match ─────▶   2
              3  ◀───── match ─────▶   3
              4   (no orders)
                                       5   (unknown customer)

  INNER  →  1, 1, 2, 3                 only the matches
  LEFT   →  1, 1, 2, 3, 4(NaN)         all customers
  RIGHT  →  1, 1, 2, 3, 5(NaN)         all orders
  OUTER  →  1, 1, 2, 3, 4(NaN), 5(NaN) everything
```

Venn-style picture (L = left keys, R = right keys):
```text
   INNER            LEFT             RIGHT            OUTER
  ( L (█) R )     (█L█(█) R )      ( L (█)█R█)      (█L█(█)█R█)
   only overlap    all of L          all of R         everything
```

In [ ]:
summary = pd.DataFrame({
    how: [len(pd.merge(customers, orders, on='Customer_ID', how=how)),
          sorted(pd.merge(customers, orders, on='Customer_ID', how=how)['Customer_ID'].unique().tolist())]
    for how in ['inner', 'left', 'right', 'outer']
}, index=['Rows', 'Customer_IDs present']).T
summary

---
<a id="sec-13"></a>
# 9. Merge on Different Column Names — `left_on` / `right_on`

### When
The same key has different names in each table (`customer_id` vs `id`) — very common when tables come from different systems.

```python
pd.merge(orders, customers, left_on='customer_id', right_on='id', how='left')
```
Both key columns are kept in the result; drop the redundant one afterwards.

In [ ]:
orders_sys = pd.DataFrame({'order_no': [1, 2, 3], 'customer_id': [10, 11, 10], 'total': [250, 900, 400]})
crm = pd.DataFrame({'id': [10, 11, 12], 'full_name': ['Ali Khan', 'Sara Ahmed', 'Omar Farooq']})
merged9 = pd.merge(orders_sys, crm, left_on='customer_id', right_on='id', how='left').drop(columns='id')
merged9

### Common mistakes
Forgetting that both columns remain (duplicate information) · swapping `left_on`/`right_on` so they refer to the wrong table.

---
<a id="sec-14"></a>
# 10. Multiple Keys (composite keys)

### What
Sometimes **one column is not enough** to identify a row. A **composite key** combines several columns, e.g. (`Customer_ID`, `Product_ID`) — one customer can have many products and one product many customers, but the **pair** identifies one row.

```python
pd.merge(df1, df2, on=['Customer_ID', 'Product_ID'])
```

In [ ]:
purchases = pd.DataFrame({'Customer_ID': [1, 1, 2, 2], 'Product_ID': ['P1', 'P2', 'P1', 'P3'], 'Qty': [2, 1, 5, 1]})
reviews = pd.DataFrame({'Customer_ID': [1, 2, 2], 'Product_ID': ['P1', 'P1', 'P2'], 'Stars': [5, 3, 4]})

print('Composite key (Customer_ID, Product_ID):')
print(pd.merge(purchases, reviews, on=['Customer_ID', 'Product_ID'], how='left'), '\n')
print('❌ Only Customer_ID — wrong pairs and extra rows:')
print(pd.merge(purchases, reviews, on='Customer_ID', how='left'))

### Explanation
With the composite key, each purchase gets **its own** review (or NaN). Merging on `Customer_ID` alone pairs every purchase of a customer with every review of that customer → wrong combinations and more rows.

---
<a id="sec-15"></a>
# 11. Merge with Suffixes

### What
If both tables have a **non-key column with the same name** (e.g. `City` in customers and `City` in orders = delivery city), pandas renames them with suffixes. The default is `_x`/`_y` — unclear! Set your own:

```python
pd.merge(customers, orders, on='Customer_ID', suffixes=('_customer', '_order'))
```

In [ ]:
cust11 = pd.DataFrame({'Customer_ID': [1, 2], 'City': ['Lahore', 'Karachi'], 'Date': ['2024-01-10', '2024-03-05']})
ord11 = pd.DataFrame({'Customer_ID': [1, 1, 2], 'City': ['Lahore', 'Islamabad', 'Karachi'], 'Date': ['2025-02-01', '2025-02-15', '2025-03-01']})
print('Default suffixes:'); print(pd.merge(cust11, ord11, on='Customer_ID'), '\n')
m11 = pd.merge(cust11, ord11, on='Customer_ID', suffixes=('_customer', '_order'))
print('Custom suffixes:'); print(m11, '\n')
print('Orders delivered outside the home city:')
print(m11[m11['City_customer'] != m11['City_order']])

### Common mistakes
Leaving `_x`/`_y` in final reports · not noticing that a column you expected (`City`) no longer exists after the merge (it's now `City_x`).

---
<a id="sec-16"></a>
# 12. Relationship Types: One-to-One, One-to-Many, Many-to-One, Many-to-Many

| Relationship | Meaning | Example | Result rows |
|---|---|---|---|
| **One-to-one (1:1)** | Each key appears **once** on both sides | Employee ↔ Employee_Details | ≈ same as the inputs |
| **One-to-many (1:m)** | Key unique on the **left**, repeated on the right | Customer (1) → Orders (many) | = number of right rows (matched) |
| **Many-to-one (m:1)** | Key repeated on the left, unique on the right | Orders (many) → Product (1) — the most common "lookup" merge | = number of left rows |
| **Many-to-many (m:m)** | Key repeated on **both** sides | Students ↔ Courses via a shared "Class" code | **Multiplies**: each left copy × each right copy |

⚠️ In a many-to-many merge, a key appearing 3 times on the left and 4 times on the right produces **3 × 4 = 12 rows** for that key. Usually this is a **bug** (a key you thought was unique isn't).

In [ ]:
emp = pd.DataFrame({'Emp_ID': [1, 2, 3], 'Name': ['Ali', 'Sara', 'Omar']})
emp_details = pd.DataFrame({'Emp_ID': [1, 2, 3], 'Salary': [90000, 85000, 70000]})
products = pd.DataFrame({'Product_ID': ['P1', 'P2'], 'Price': [500, 1200]})
order_lines = pd.DataFrame({'Order_ID': [1, 2, 3, 4], 'Product_ID': ['P1', 'P1', 'P2', 'P1']})
teachers = pd.DataFrame({'Class': ['A', 'A', 'B'], 'Teacher': ['Mr. Khan', 'Ms. Noor', 'Mr. Ali']})
pupils = pd.DataFrame({'Class': ['A', 'A', 'A', 'B'], 'Pupil': ['Hina', 'Bilal', 'Zara', 'Usman']})

print('1:1 →', len(pd.merge(emp, emp_details, on='Emp_ID')), 'rows (3 employees)')
print('1:m →', len(pd.merge(customers, orders, on='Customer_ID')), 'rows (one per matched order)')
print('m:1 →', len(pd.merge(order_lines, products, on='Product_ID')), 'rows (one per order line)')
mm = pd.merge(teachers, pupils, on='Class')
print('m:m →', len(mm), 'rows = class A: 2 teachers × 3 pupils = 6, class B: 1 × 1 = 1')
mm

---
<a id="sec-17"></a>
# 13. Validate the Merge — `validate=`

### What
Tells pandas the relationship you **expect**; it raises a `MergeError` if the data disagrees.

| Value | Checks |
|---|---|
| `'one_to_one'` / `'1:1'` | Keys unique on **both** sides |
| `'one_to_many'` / `'1:m'` | Keys unique on the **left** |
| `'many_to_one'` / `'m:1'` | Keys unique on the **right** (typical lookup) |
| `'many_to_many'` / `'m:m'` | No check (documents intent) |

### Why it helps
A product table that accidentally contains `P1` twice would silently **duplicate every order line** of P1 and inflate revenue. `validate='many_to_one'` stops the merge immediately and tells you why.

In [ ]:
print('✔ OK:', len(pd.merge(order_lines, products, on='Product_ID', validate='many_to_one')), 'rows')

products_bad = pd.concat([products, pd.DataFrame({'Product_ID': ['P1'], 'Price': [550]})], ignore_index=True)   # P1 twice!
print('Without validation → silently', len(pd.merge(order_lines, products_bad, on='Product_ID')), 'rows instead of 4')
try:
    pd.merge(order_lines, products_bad, on='Product_ID', validate='many_to_one')
except pd.errors.MergeError as e:
    print('With validate="many_to_one" → MergeError:', str(e).split('\n')[0])

### Common mistakes
Never validating lookup merges · choosing the wrong direction (`one_to_many` vs `many_to_one` depends on which table is **left**).

---
<a id="sec-18"></a>
# 14. Merge Indicator — `indicator=True`

Adds a `_merge` column telling where each row came from:

| `_merge` value | Meaning |
|---|---|
| `both` | Key found in both tables |
| `left_only` | Only in the left table (e.g. customers without orders) |
| `right_only` | Only in the right table (e.g. orders with an unknown customer → data-quality problem!) |

Use it with `how='outer'` to **reconcile** two tables.

In [ ]:
check = pd.merge(customers, orders, on='Customer_ID', how='outer', indicator=True)
print(check[['Customer_ID', 'Name', 'Order_ID', '_merge']], '\n')
print(check['_merge'].value_counts(), '\n')
print('Customers with no orders :', check.loc[check['_merge'] == 'left_only', 'Name'].tolist())
print('Orders with unknown customer:', check.loc[check['_merge'] == 'right_only', 'Order_ID'].astype(int).tolist())

# Anti-join: rows of the left table that have NO match in the right table
no_orders = (customers.merge(orders[['Customer_ID']].drop_duplicates(), on='Customer_ID', how='left', indicator=True)
             .query('_merge == "left_only"').drop(columns='_merge'))
print('\nAnti-join result:'); print(no_orders)

---
<a id="sec-19"></a>
# 15. `df.join()`

### What
A convenience method that combines DataFrames **using the index** (row labels). By default it's a **left** join on the index.

```python
left.join(right)                               # left index ↔ right index, how='left'
left.join(right, how='inner' / 'outer' / 'right')
left.join(right, on='Customer_ID')             # left COLUMN ↔ right INDEX
left.join(right, lsuffix='_l', rsuffix='_r')   # needed if column names overlap
left.join([df2, df3])                          # several tables at once (all on the index)
```

### Why / When
Tables that are already **indexed by their key** (e.g. results of `groupby`, `set_index('Customer_ID')`, time-indexed data).

### When not
Joining on ordinary columns on both sides → `merge` is clearer.

In [ ]:
cust_idx = customers.set_index('Customer_ID')                                    # index = Customer_ID
totals = orders.groupby('Customer_ID')['Amount'].agg(Total_Spent='sum', Orders='count')   # index = Customer_ID

print("join (default how='left'):"); print(cust_idx.join(totals), '\n')
print("join how='inner':"); print(cust_idx.join(totals, how='inner'), '\n')
print("join how='outer' (customer 5 appears with NaN name):"); print(cust_idx.join(totals, how='outer'), '\n')

print("orders.join(cust_idx, on='Customer_ID') — column on the left, index on the right:")
print(orders.join(cust_idx, on='Customer_ID'))

### Common mistakes
Forgetting that `join` uses the **index** of the right table (join on a column that isn't set as the index → wrong or no matches) · overlapping column names without `lsuffix`/`rsuffix` → `ValueError`.

---
<a id="sec-20"></a>
# 16. `merge` vs `join` vs `concat`

| Method | Main purpose | Uses key columns? | Uses index? | Default | Common use |
|---|---|---|---|---|---|
| `pd.concat` | **Stack/glue** DataFrames along an axis | ❌ No | ✅ aligns on index (axis=1) / columns (axis=0) | outer, axis=0 | Monthly files, adding rows, side-by-side columns with the same index |
| `pd.merge` / `df.merge` | **Match rows by key values** (SQL-style) | ✅ `on`, `left_on`, `right_on` | Optional (`left_index`, `right_index`) | inner | Customers ↔ orders ↔ products |
| `df.join` | Match rows **by index** (shortcut for merge) | Only on the left (`on=`) | ✅ right side always index | left | Adding groupby results or indexed lookup tables |

**In one sentence each:**
- **concat** = "put these tables **together**" (no matching).
- **merge** = "**match** these tables using these columns".
- **join** = "match using the **row labels**".

### Same data, three methods

In [ ]:
scores = pd.DataFrame({'Student': ['Ali', 'Sara', 'Omar'], 'Math': [85, 92, 78]})
extra = pd.DataFrame({'Student': ['Omar', 'Ali', 'Hina'], 'English': [70, 88, 95]})     # different ORDER, one new student

print('concat axis=0 (stack rows — no matching!):')
print(pd.concat([scores, extra], ignore_index=True), '\n')
print('concat axis=1 (side by side by POSITION/index — WRONG pairing: Ali gets Omar\'s English):')
print(pd.concat([scores, extra], axis=1), '\n')
print("merge on 'Student' (correct pairing by name):")
print(pd.merge(scores, extra, on='Student', how='outer'), '\n')
print('join on index after set_index (same as merge):')
print(scores.set_index('Student').join(extra.set_index('Student'), how='outer'))

### Explanation
Only `merge` (or `join` after setting the key as the index) pairs **Ali's Math with Ali's English**. `concat(axis=1)` blindly pairs row 0 with row 0 — a silent error because the second table is in a different order.

---
<a id="sec-21"></a>
# 17. Handling Duplicate Keys

### What happens
If a key that should be unique appears several times, every copy matches every copy on the other side → **row multiplication**, double-counted revenue, inflated averages.

### How to detect
```python
df['key'].duplicated().sum()          # how many repeated keys
df[df['key'].duplicated(keep=False)]  # show all copies
df['key'].value_counts().head()       # keys with count > 1
df['key'].is_unique                   # True/False
```

In [ ]:
prices = pd.DataFrame({'Product_ID': ['P1', 'P2', 'P3', 'P2'], 'Price': [500, 1200, 300, 1250]})   # P2 twice (price update not cleaned)
sales = pd.DataFrame({'Sale_ID': range(1, 7), 'Product_ID': ['P1', 'P2', 'P2', 'P3', 'P2', 'P1'], 'Qty': [1, 2, 1, 3, 1, 2]})

print('Is Product_ID unique in prices?', prices['Product_ID'].is_unique)
print('value_counts:'); print(prices['Product_ID'].value_counts().head(3))
print('Duplicated rows:'); print(prices[prices['Product_ID'].duplicated(keep=False)], '\n')

bad = sales.merge(prices, on='Product_ID')
print(f'Sales rows: {len(sales)} → after merge: {len(bad)}  ← row multiplication!')
print('Revenue (wrong):', (bad['Qty'] * bad['Price']).sum())

prices_clean = prices.drop_duplicates(subset='Product_ID', keep='last')                  # keep the latest price
good = sales.merge(prices_clean, on='Product_ID', validate='many_to_one')
print('Revenue (correct, one price per product):', (good['Qty'] * good['Price']).sum(), '| rows:', len(good))

### Explanation
Each of the 3 P2 sales matched **both** P2 prices → 6 rows instead of 3 and revenue that is too high. After deduplicating (and validating), row counts match again. Which duplicate to keep is a **business decision** (latest price? highest?).

---
<a id="sec-22"></a>
# 18. Preventing Bad Merges — Checklist

| Check | How | Why |
|---|---|---|
| **Key uniqueness** | `df['key'].is_unique`, `duplicated().sum()` | Avoid row multiplication |
| **Missing keys** | `df['key'].isna().sum()` | NaN keys never match (or match each other unexpectedly) |
| **Data types** | `df1['key'].dtype == df2['key'].dtype` | `1` (int) ≠ `'1'` (text) → no matches or a `ValueError` |
| **Formatting** | strip spaces, same case, same leading zeros (`'007'` vs `'7'`) | Hidden mismatches |
| **Row counts** | `len()` before vs after | Detect lost rows (inner) or multiplied rows (duplicates) |
| **Unexpected nulls** | `merged.isna().sum()` after a left join | Keys that didn't match |
| **Relationship** | `validate='many_to_one'` etc. | Fail fast when assumptions break |
| **Unmatched records** | `indicator=True` | Find left_only / right_only rows |

In [ ]:
ord18 = pd.DataFrame({'Order_ID': [1, 2, 3, 4], 'Customer_ID': ['C01', 'c02 ', 'C03', None]})
cus18 = pd.DataFrame({'Customer_ID': ['C01', 'C02', 'C03', 'C04'], 'Name': ['Ali', 'Sara', 'Omar', 'Hina']})

def merge_report(left, right, key, how='left', **kw):
    '''Merge with a quick quality report.'''
    print(f'left rows={len(left)}, right rows={len(right)} | dtypes: {left[key].dtype} vs {right[key].dtype}')
    print(f'missing keys: left={left[key].isna().sum()}, right={right[key].isna().sum()} | right key unique: {right[key].is_unique}')
    out = pd.merge(left, right, on=key, how=how, indicator=True, **kw)
    print(f'result rows={len(out)} | match status: {out["_merge"].value_counts().to_dict()}')
    return out

print('BEFORE cleaning:'); r1 = merge_report(ord18, cus18, 'Customer_ID', validate='many_to_one')
ord18['Customer_ID'] = ord18['Customer_ID'].str.strip().str.upper()                    # fix formatting
print('\nAFTER cleaning:'); r2 = merge_report(ord18, cus18, 'Customer_ID', validate='many_to_one')
r2

In [ ]:
# Data-type mismatch: integers vs strings
a = pd.DataFrame({'ID': [1, 2, 3], 'x': ['a', 'b', 'c']})
b = pd.DataFrame({'ID': ['1', '2', '3'], 'y': [10, 20, 30]})
try:
    pd.merge(a, b, on='ID')
except ValueError as e:
    print('ValueError:', str(e)[:80], '...')
b['ID'] = b['ID'].astype(int)                                  # convert to the same type
print(pd.merge(a, b, on='ID'))

---
<a id="sec-23"></a>
# 19. Combining Real-World Tables: Customers + Orders + Products

Goal: one analytical DataFrame with every order enriched with customer and product details.

```text
orders ──(m:1 on Customer_ID)──▶ customers
   │
   └──(m:1 on Product_ID)──▶ products
```
Start from the **fact table** (orders — one row per event) and **left-join** the lookup tables, validating `many_to_one` each time.

In [ ]:
customers19 = pd.DataFrame({'Customer_ID': [1, 2, 3, 4, 5], 'Name': ['Ali', 'Sara', 'Omar', 'Hina', 'Bilal'],
                            'City': ['Lahore', 'Karachi', 'Lahore', 'Islamabad', 'Karachi'], 'Age': [28, 34, 45, 23, 39]})
products19 = pd.DataFrame({'Product_ID': ['P1', 'P2', 'P3', 'P4'], 'Product': ['Laptop', 'Mouse', 'Desk', 'Lamp'],
                           'Category': ['Electronics', 'Electronics', 'Furniture', 'Furniture'], 'Price': [150000, 2500, 30000, 4500]})
orders19 = pd.DataFrame({'Order_ID': range(1001, 1009), 'Customer_ID': [1, 2, 1, 3, 5, 2, 4, 1],
                         'Product_ID': ['P1', 'P2', 'P2', 'P3', 'P4', 'P1', 'P4', 'P3'], 'Quantity': [1, 3, 2, 1, 2, 1, 4, 1]})

analytics = (orders19
             .merge(customers19, on='Customer_ID', how='left', validate='many_to_one')
             .merge(products19, on='Product_ID', how='left', validate='many_to_one'))
analytics['Revenue'] = analytics['Quantity'] * analytics['Price']
print(f'orders: {len(orders19)} rows → analytics: {len(analytics)} rows (must be equal) | nulls: {analytics.isna().sum().sum()}')
print(analytics, '\n')
print('Revenue by city:'); print(analytics.groupby('City')['Revenue'].sum().sort_values(ascending=False))
print('\nRevenue by category:'); print(analytics.groupby('Category')['Revenue'].sum())

---
<a id="sec-24"></a>
# 20. Advanced Combining

<a id="sec-25"></a>
## 20.1 Merge multiple DataFrames
Chain `.merge()` calls (as above), or use `functools.reduce` when you have a list of tables sharing the same key.

In [ ]:
from functools import reduce
math = pd.DataFrame({'Student': ['Ali', 'Sara', 'Omar'], 'Math': [85, 92, 78]})
eng = pd.DataFrame({'Student': ['Ali', 'Sara', 'Omar'], 'English': [70, 88, 81]})
sci = pd.DataFrame({'Student': ['Ali', 'Sara', 'Omar'], 'Science': [90, 75, 84]})
reduce(lambda left, right: pd.merge(left, right, on='Student', validate='one_to_one'), [math, eng, sci])

<a id="sec-26"></a>
## 20.2 Merge after aggregation
Aggregate the "many" side **first**, then merge → no row multiplication, one summary row per customer.

In [ ]:
per_customer = analytics.groupby('Customer_ID').agg(Orders=('Order_ID', 'count'), Total_Revenue=('Revenue', 'sum')).reset_index()
customer_profile = customers19.merge(per_customer, on='Customer_ID', how='left', validate='one_to_one')
customer_profile[['Orders', 'Total_Revenue']] = customer_profile[['Orders', 'Total_Revenue']].fillna(0)
customer_profile['Share_%'] = (customer_profile['Total_Revenue'] / customer_profile['Total_Revenue'].sum() * 100).round(1)
customer_profile

<a id="sec-27"></a>
## 20.3 Merge with indexes, 20.4 MultiIndex joins

In [ ]:
prod_idx = products19.set_index('Product_ID')
print('merge with right_index=True:')
print(pd.merge(orders19.head(3), prod_idx, left_on='Product_ID', right_index=True), '\n')

# MultiIndex: sales per (City, Category) joined with targets per (City, Category)
actual = analytics.groupby(['City', 'Category'])['Revenue'].sum()
targets = pd.DataFrame({'City': ['Lahore', 'Lahore', 'Karachi', 'Karachi', 'Islamabad'],
                        'Category': ['Electronics', 'Furniture', 'Electronics', 'Furniture', 'Furniture'],
                        'Target': [300000, 40000, 150000, 10000, 20000]}).set_index(['City', 'Category'])
print('MultiIndex join on (City, Category):')
print(targets.join(actual, how='left').assign(Achieved_pct=lambda d: (d['Revenue'] / d['Target'] * 100).round(0)))

<a id="sec-28"></a>
## 20.5 `pd.merge_asof()` — nearest-key (time) matching
Matches each left row to the **most recent** (or nearest) right row by a sorted key — e.g. attach the exchange rate that was valid at the time of each transaction. Both tables must be **sorted** by the key.

<a id="sec-29"></a>
## 20.6 `pd.merge_ordered()` — ordered outer merge with filling
Merges ordered data (like dates) keeping order, with optional forward fill (`fill_method='ffill'`).

In [ ]:
trades = pd.DataFrame({'Time': pd.to_datetime(['2025-03-01 10:02', '2025-03-01 10:07', '2025-03-01 10:15', '2025-03-01 10:31']),
                       'Amount_USD': [100, 250, 80, 500]})
rates = pd.DataFrame({'Time': pd.to_datetime(['2025-03-01 10:00', '2025-03-01 10:10', '2025-03-01 10:30']),
                      'PKR_per_USD': [279.5, 280.1, 280.6]})
asof = pd.merge_asof(trades.sort_values('Time'), rates.sort_values('Time'), on='Time', direction='backward')
asof['Amount_PKR'] = asof['Amount_USD'] * asof['PKR_per_USD']
print('merge_asof (latest rate at or before each trade):'); print(asof, '\n')

stock_a = pd.DataFrame({'Date': pd.to_datetime(['2025-01-01', '2025-01-03', '2025-01-05']), 'A_Price': [100, 102, 101]})
stock_b = pd.DataFrame({'Date': pd.to_datetime(['2025-01-02', '2025-01-03', '2025-01-04']), 'B_Price': [50, 51, 53]})
print('merge_ordered with forward fill:')
print(pd.merge_ordered(stock_a, stock_b, on='Date', fill_method='ffill'))

<a id="sec-30"></a>
## 20.7 Combining time-series data & 20.8 concatenating many files/DataFrames

In [ ]:
# Time series from two sources on a shared DatetimeIndex → join (index-based)
idx = pd.date_range('2025-01-01', periods=6, freq='D')
web = pd.DataFrame({'Web_Visits': [120, 135, 128, 150, 160, 155]}, index=idx)
store = pd.DataFrame({'Store_Sales': [30, 32, 29, 40]}, index=idx[[0, 1, 3, 5]])     # some days missing
combined_ts = web.join(store, how='left')
print(combined_ts, '\n')

# Many "files": build a LIST, then concat ONCE (never concat inside the loop)
import os, tempfile
folder = tempfile.mkdtemp()
for month in ['2025-01', '2025-02', '2025-03']:
    pd.DataFrame({'Order_ID': np.arange(3) + int(month[-2:]) * 100, 'Amount': [100, 200, 300]}).to_csv(f'{folder}/sales_{month}.csv', index=False)

pieces = []
for file in sorted(os.listdir(folder)):
    part = pd.read_csv(os.path.join(folder, file))
    part['Source_File'] = file                       # remember the origin
    pieces.append(part)
all_sales = pd.concat(pieces, ignore_index=True)
print(all_sales)

---
<a id="sec-31"></a>
# 21. Common Mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| **Wrong join type** | Rows silently disappear (inner) or unexpected NaNs (outer) | Decide which table is the "main" one; usually `how='left'` from the fact table; compare row counts |
| **Wrong key** | Nonsense pairs, extra rows | Merge on the true identifier (or a composite key) |
| **Duplicate keys** | Row multiplication, inflated totals | `is_unique`, `duplicated()`, `validate=` |
| **Different data types** | `ValueError` or zero matches | `astype()` so both keys have the same type |
| **Missing / inconsistent keys** | Unmatched rows, NaN columns | Clean formatting (strip, upper, leading zeros); `indicator=True` |
| **Unexpected row multiplication** | `len(result) > len(fact_table)` after a lookup merge | Deduplicate the lookup table; aggregate before merging |
| **Forgetting `ignore_index`** | Duplicate index labels after concat; `.loc` returns several rows | `pd.concat(..., ignore_index=True)` |
| **Confusing `concat` with `merge`** | Rows paired by position instead of by ID | Use `merge` whenever rows must be **matched** |
| **Using the index incorrectly** | `join`/`concat(axis=1)` misaligned; NaNs | Check/`set_index` the right key; `reset_index` when needed |
| **Default suffixes `_x`/`_y`** | Confusing column names | `suffixes=('_customer', '_order')` |
| **concat in a loop** | Very slow on many files | Collect in a list, concat once |

---
# Practice Problems (20, increasing difficulty)

Run the next cell to create the practice datasets. Each problem has an empty code cell; solutions are in the **Solutions** section.

In [ ]:
# ---------- Practice datasets ----------
q_jan = pd.DataFrame({'Sale_ID': [1, 2, 3], 'Store': ['A', 'B', 'A'], 'Amount': [500, 700, 300]})
q_feb = pd.DataFrame({'Sale_ID': [4, 5], 'Store': ['B', 'C'], 'Amount': [650, 900]})
q_mar = pd.DataFrame({'Sale_ID': [6, 7], 'Store': ['A', 'C'], 'Amount': [400, 1000], 'Promo': ['Yes', 'No']})

students_p = pd.DataFrame({'Student_ID': [1, 2, 3, 4, 5], 'Name': ['Ali', 'Sara', 'Omar', 'Hina', 'Bilal'],
                           'Class': ['A', 'A', 'B', 'B', 'C']})
marks_p = pd.DataFrame({'Student_ID': [1, 2, 2, 3, 6], 'Subject': ['Math', 'Math', 'English', 'Math', 'Math'],
                        'Marks': [88, 92, 75, 64, 71]})
classes_p = pd.DataFrame({'class_code': ['A', 'B', 'D'], 'Teacher': ['Mr. Khan', 'Ms. Noor', 'Mr. Ali']})

emp_p = pd.DataFrame({'Emp_ID': [10, 11, 12, 13], 'Name': ['Usman', 'Zara', 'Imran', 'Ayesha'], 'Dept_ID': [1, 2, 1, 3]})
dept_p = pd.DataFrame({'Dept_ID': [1, 2, 3, 3], 'Dept': ['IT', 'HR', 'Sales', 'Sales & Marketing']})    # Dept 3 twice!

stock_p = pd.DataFrame({'Store': ['A', 'A', 'B', 'B'], 'Product': ['Pen', 'Book', 'Pen', 'Bag'], 'Stock': [50, 20, 35, 10]})
price_p = pd.DataFrame({'Store': ['A', 'A', 'B', 'B'], 'Product': ['Pen', 'Book', 'Pen', 'Lamp'], 'Price': [20, 300, 22, 900]})

orders_p = pd.DataFrame({'order_id': range(1, 9), 'cust': ['C1', 'C2', 'c3', 'C1', 'C4 ', 'C2', 'C9', 'C3'],
                         'prod': [101, 102, 101, 103, 102, 104, 101, 103], 'qty': [1, 2, 1, 3, 1, 2, 1, 1]})
customers_p = pd.DataFrame({'id': ['C1', 'C2', 'C3', 'C4', 'C5'], 'name': ['Ali', 'Sara', 'Omar', 'Hina', 'Bilal'],
                            'city': ['Lahore', 'Karachi', 'Lahore', 'Multan', 'Karachi']})
products_p = pd.DataFrame({'prod_id': ['101', '102', '103', '104'], 'product': ['Pen', 'Book', 'Bag', 'Lamp'],
                           'price': [20, 300, 1200, 900]})
rates_p = pd.DataFrame({'Date': pd.to_datetime(['2025-01-01', '2025-01-10', '2025-01-20']), 'Rate': [279.0, 280.2, 281.0]})
payments_p = pd.DataFrame({'Date': pd.to_datetime(['2025-01-05', '2025-01-12', '2025-01-25', '2025-01-10']), 'USD': [100, 50, 200, 80]})
print('Practice datasets ready ✔')

### ✍️ Practice Problem 1

**(Easy) Row-wise concat:** Stack `q_jan` and `q_feb` into one DataFrame with a clean 0..n−1 index.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 1 — write your solution here

### ✍️ Practice Problem 2

**(Easy) keys:** Stack `q_jan`, `q_feb`, `q_mar` with `keys=['Jan', 'Feb', 'Mar']`, then turn the key level into a `Month` column. What happens to the `Promo` column for January and February?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 2 — write your solution here

### ✍️ Practice Problem 3

**(Easy) join='inner':** Stack the three months keeping **only** the columns shared by all of them.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 3 — write your solution here

### ✍️ Practice Problem 4

**(Easy) Column-wise concat:** Create `a = pd.DataFrame({'x': [1, 2, 3]})` and `b = pd.DataFrame({'y': [10, 20, 30]}, index=[1, 2, 3])`. Concatenate them with `axis=1` and explain the result. Fix it so rows are paired by position.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 4 — write your solution here

### ✍️ Practice Problem 5

**(Easy) Inner merge:** Merge `students_p` with `marks_p` on `Student_ID` (inner). Which students/marks are missing and why?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 5 — write your solution here

### ✍️ Practice Problem 6

**(Easy) Left merge:** Keep **all** students, attaching their marks. Which students have no marks?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 6 — write your solution here

### ✍️ Practice Problem 7

**(Medium) Right & outer:** Do a right and an outer merge of `students_p` and `marks_p`. Report the row count of inner, left, right and outer joins in a small table.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 7 — write your solution here

### ✍️ Practice Problem 8

**(Medium) Indicator:** Use an outer merge with `indicator=True` to list (a) students without marks and (b) marks for unknown students.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 8 — write your solution here

### ✍️ Practice Problem 9

**(Medium) Different key names:** Attach each student's teacher: `students_p.Class` ↔ `classes_p.class_code`. Keep all students and drop the redundant key column.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 9 — write your solution here

### ✍️ Practice Problem 10

**(Medium) Composite key:** Merge `stock_p` and `price_p` on both `Store` and `Product` (outer). Then show what goes wrong if you merge on `Product` only.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 10 — write your solution here

### ✍️ Practice Problem 11

**(Medium) Suffixes:** Create `x = pd.DataFrame({'ID': [1, 2], 'Score': [80, 90]})` and `y = pd.DataFrame({'ID': [1, 2], 'Score': [85, 70]})` (midterm vs final). Merge them with meaningful suffixes and add an `Improvement` column.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 11 — write your solution here

### ✍️ Practice Problem 12

**(Medium) Relationship & validation:** Merge `emp_p` with `dept_p` on `Dept_ID`. How many rows do you get and why? Re-run with `validate='many_to_one'`, catch the error, then fix `dept_p` (keep the first name) and merge again.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 12 — write your solution here

### ✍️ Practice Problem 13

**(Medium) join():** Set `Student_ID` as the index of `students_p`. Compute each student's average mark from `marks_p` with `groupby` and attach it with `join()` (keep all students). Then use `marks_p.join(..., on='Student_ID')` to add student names to every mark.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 13 — write your solution here

### ✍️ Practice Problem 14

**(Medium) merge vs concat:** Using `students_p` and `marks_p`, show why `pd.concat([students_p, marks_p], axis=1)` is the wrong way to combine them.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 14 — write your solution here

### ✍️ Practice Problem 15

**(Hard) Duplicate-key detection:** For `dept_p`, `marks_p` and `orders_p`, report which key columns are unique (`is_unique`) and list duplicated keys with `value_counts()`. Which ones are problems for a lookup merge?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 15 — write your solution here

### ✍️ Practice Problem 16

**(Hard) Cleaning keys:** Merge `orders_p` with `customers_p` (`cust` ↔ `id`). First show how many orders fail to match; then clean the keys (strip + upper) and merge again with `validate='many_to_one'` and `indicator=True`.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 16 — write your solution here

### ✍️ Practice Problem 17

**(Hard) Data-type mismatch:** Merge the cleaned orders with `products_p` (`prod` ↔ `prod_id`). Fix the type mismatch and compute `revenue = qty × price`.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 17 — write your solution here

### ✍️ Practice Problem 18

**(Hard) Merge after aggregation:** Build a customer summary (orders, total quantity, total revenue) by aggregating first and then merging onto `customers_p` (all customers, zeros for those without orders).

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 18 — write your solution here

### ✍️ Practice Problem 19

**(Hard) merge_asof:** For each payment in `payments_p`, attach the latest exchange rate valid **on or before** the payment date from `rates_p`, and compute the PKR amount.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 19 — write your solution here

### ✍️ Practice Problem 20

**(Hard) Full pipeline with checks:** Write a function `safe_lookup(left, right, left_key, right_key)` that prints row counts, checks key dtypes and uniqueness on the right, merges with `how='left'`, `validate='many_to_one'`, `indicator=True`, and warns about unmatched rows. Use it to build the full orders + customers + products table.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 20 — write your solution here

---
---
# Solutions

⚠️ Try the problems first!

### Solution 1

In [ ]:
pd.concat([q_jan, q_feb], ignore_index=True)

**Explanation:** `ignore_index=True` replaces the repeated 0, 1, 2 / 0, 1 labels with 0..4.

### Solution 2

In [ ]:
months = (pd.concat([q_jan, q_feb, q_mar], keys=['Jan', 'Feb', 'Mar'], names=['Month', 'Row'])
          .reset_index(level='Month').reset_index(drop=True))
months

**Explanation:** `Promo` only exists in March, so it is NaN for the January and February rows (outer join of columns).

### Solution 3

In [ ]:
pd.concat([q_jan, q_feb, q_mar], join='inner', ignore_index=True)

**Explanation:** `join='inner'` drops `Promo` because it is not present in every piece.

### Solution 4

In [ ]:
a = pd.DataFrame({'x': [1, 2, 3]})
b = pd.DataFrame({'y': [10, 20, 30]}, index=[1, 2, 3])
print(pd.concat([a, b], axis=1))                                     # aligned by index labels 0..3 → NaNs
print(pd.concat([a, b.reset_index(drop=True)], axis=1))             # paired by position

**Explanation:** `axis=1` aligns on index labels: label 0 exists only in `a`, label 3 only in `b` → 4 rows with NaNs. Resetting `b`'s index pairs rows by position.

### Solution 5

In [ ]:
inner5 = pd.merge(students_p, marks_p, on='Student_ID', how='inner')
inner5

**Explanation:** Hina (4) and Bilal (5) have no marks → dropped. The mark for Student_ID 6 has no student → dropped. Sara appears twice (two subjects).

### Solution 6

In [ ]:
left6 = pd.merge(students_p, marks_p, on='Student_ID', how='left')
print(left6)
print('No marks:', left6.loc[left6['Marks'].isna(), 'Name'].tolist())

**Explanation:** All five students are kept; Hina and Bilal get NaN for Subject/Marks.

### Solution 7

In [ ]:
counts7 = {how: len(pd.merge(students_p, marks_p, on='Student_ID', how=how)) for how in ['inner', 'left', 'right', 'outer']}
print(pd.Series(counts7, name='rows'))
print(pd.merge(students_p, marks_p, on='Student_ID', how='outer'))

**Explanation:** inner 4 · left 6 (4 matches + Hina + Bilal) · right 5 (every mark, including ID 6) · outer 7 (everything).

### Solution 8

In [ ]:
chk8 = pd.merge(students_p, marks_p, on='Student_ID', how='outer', indicator=True)
print('Students without marks:', chk8.loc[chk8['_merge'] == 'left_only', 'Name'].tolist())
print('Marks for unknown students:'); print(chk8.loc[chk8['_merge'] == 'right_only', ['Student_ID', 'Subject', 'Marks']])

**Explanation:** `left_only` = only in students; `right_only` = only in marks (Student_ID 6 → a data-quality issue).

### Solution 9

In [ ]:
pd.merge(students_p, classes_p, left_on='Class', right_on='class_code', how='left').drop(columns='class_code')

**Explanation:** Bilal's class C has no teacher (NaN); class D in `classes_p` has no students and is ignored by the left join.

### Solution 10

In [ ]:
print(pd.merge(stock_p, price_p, on=['Store', 'Product'], how='outer'))
print('\n❌ Product only:')
print(pd.merge(stock_p, price_p, on='Product', how='outer'))

**Explanation:** The composite key pairs each store's product with that store's price. On `Product` alone, the Pen rows of store A and B cross-match (2 × 2 = 4 rows) and `Store_x`/`Store_y` show the mix-up.

### Solution 11

In [ ]:
x = pd.DataFrame({'ID': [1, 2], 'Score': [80, 90]})
y = pd.DataFrame({'ID': [1, 2], 'Score': [85, 70]})
m11 = pd.merge(x, y, on='ID', suffixes=('_midterm', '_final'), validate='one_to_one')
m11['Improvement'] = m11['Score_final'] - m11['Score_midterm']
m11

**Explanation:** Meaningful suffixes make the merged columns self-explanatory.

### Solution 12

In [ ]:
print('Rows:', len(pd.merge(emp_p, dept_p, on='Dept_ID')), '(4 employees, but Ayesha appears twice)')
try:
    pd.merge(emp_p, dept_p, on='Dept_ID', validate='many_to_one')
except pd.errors.MergeError as e:
    print('MergeError:', str(e).split('\n')[0])
dept_fixed = dept_p.drop_duplicates(subset='Dept_ID', keep='first')
pd.merge(emp_p, dept_fixed, on='Dept_ID', validate='many_to_one')

**Explanation:** Dept_ID 3 appears twice, so Ayesha matches both rows (5 rows). Validation catches it; deduplicating the lookup table fixes it.

### Solution 13

In [ ]:
st_idx = students_p.set_index('Student_ID')
avg_marks = marks_p.groupby('Student_ID')['Marks'].mean().rename('Avg_Mark')
print(st_idx.join(avg_marks))                                    # left join on the index
print(marks_p.join(st_idx[['Name']], on='Student_ID'))           # column (left) ↔ index (right)

**Explanation:** `join` matches on the index; `on=` lets the **left** side use a column while the right side uses its index.

### Solution 14

In [ ]:
print(pd.concat([students_p, marks_p], axis=1))
# Row 0 of students is glued to row 0 of marks by POSITION, not by Student_ID:
# e.g. Omar (ID 3) is paired with Sara's English mark (ID 2), and Bilal (ID 5) gets the mark of unknown student 6.

**Explanation:** `concat(axis=1)` never looks at `Student_ID`; only `merge` pairs rows by the key.

### Solution 15

In [ ]:
for name, frame, key in [('dept_p', dept_p, 'Dept_ID'), ('marks_p', marks_p, 'Student_ID'), ('orders_p', orders_p, 'cust')]:
    vc = frame[key].value_counts()
    print(f'{name}.{key}: unique={frame[key].is_unique} | duplicated keys: {vc[vc > 1].to_dict()}')

**Explanation:** `dept_p` is meant to be a lookup table (one row per department) → its duplicate is a **problem**. `marks_p` and `orders_p` are fact tables where repeated keys are **expected** (many marks per student, many orders per customer).

### Solution 16

In [ ]:
before = pd.merge(orders_p, customers_p, left_on='cust', right_on='id', how='left', indicator=True)
print('Unmatched before cleaning:', (before['_merge'] == 'left_only').sum(), before.loc[before['_merge'] == 'left_only', 'cust'].tolist())
orders_clean = orders_p.assign(cust=orders_p['cust'].str.strip().str.upper())
after = pd.merge(orders_clean, customers_p, left_on='cust', right_on='id', how='left', validate='many_to_one', indicator=True)
print('Unmatched after cleaning:', (after['_merge'] == 'left_only').sum(), after.loc[after['_merge'] == 'left_only', 'cust'].tolist())
after

**Explanation:** 'c3' and 'C4 ' failed only because of case/spaces. After cleaning, only C9 is unmatched — a customer that truly doesn't exist.

### Solution 17

In [ ]:
print(orders_clean['prod'].dtype, 'vs', products_p['prod_id'].dtype)
products_fixed = products_p.assign(prod_id=products_p['prod_id'].astype(int))
full17 = orders_clean.merge(products_fixed, left_on='prod', right_on='prod_id', how='left', validate='many_to_one')
full17['revenue'] = full17['qty'] * full17['price']
full17[['order_id', 'cust', 'product', 'qty', 'price', 'revenue']]

**Explanation:** int64 vs text keys raise a ValueError; converting `prod_id` to int makes them compatible.

### Solution 18

In [ ]:
summary18 = full17.groupby('cust').agg(orders=('order_id', 'count'), total_qty=('qty', 'sum'), revenue=('revenue', 'sum')).reset_index()
cust_summary = customers_p.merge(summary18, left_on='id', right_on='cust', how='left').drop(columns='cust')
cust_summary[['orders', 'total_qty', 'revenue']] = cust_summary[['orders', 'total_qty', 'revenue']].fillna(0).astype(int)
cust_summary

**Explanation:** Aggregating first gives exactly one row per customer, so the merge is one-to-one and cannot multiply rows. Bilal (C5) gets zeros.

### Solution 19

In [ ]:
asof19 = pd.merge_asof(payments_p.sort_values('Date'), rates_p.sort_values('Date'), on='Date', direction='backward')
asof19['PKR'] = asof19['USD'] * asof19['Rate']
asof19

**Explanation:** `merge_asof` needs both tables sorted by `Date`; `direction='backward'` picks the most recent rate on or before each payment (Jan 10 uses the Jan 10 rate).

### Solution 20

In [ ]:
def safe_lookup(left, right, left_key, right_key):
    print(f'→ {left_key} ↔ {right_key}: left rows={len(left)}, right rows={len(right)}')
    if left[left_key].dtype != right[right_key].dtype:
        print(f'  ⚠ dtype mismatch {left[left_key].dtype} vs {right[right_key].dtype} → converting right key')
        right = right.assign(**{right_key: right[right_key].astype(left[left_key].dtype)})
    if not right[right_key].is_unique:
        raise ValueError(f'right key {right_key} is not unique — fix the lookup table first')
    out = pd.merge(left, right, left_on=left_key, right_on=right_key, how='left', validate='many_to_one', indicator=True)
    unmatched = (out['_merge'] == 'left_only').sum()
    if unmatched:
        print(f'  ⚠ {unmatched} unmatched row(s): {out.loc[out["_merge"] == "left_only", left_key].tolist()}')
    assert len(out) == len(left), 'row count changed!'
    return out.drop(columns='_merge')

full20 = (orders_p.assign(cust=orders_p['cust'].str.strip().str.upper())
          .pipe(safe_lookup, customers_p, 'cust', 'id')
          .pipe(safe_lookup, products_p, 'prod', 'prod_id'))
full20['revenue'] = full20['qty'] * full20['price']
full20

**Explanation:** The helper bundles the checklist from Section 18. `.pipe` lets us chain it like a normal method.

---
<a id="sec-32"></a>
# Final Project — E-Commerce Database Integration

**Scenario:** an online store exports four tables from different systems. They contain realistic problems (duplicate keys, unknown IDs, different key names, mismatched data types, missing payments). Build one clean analytical DataFrame and analyse it.

## Create the four tables

In [ ]:
rng = np.random.default_rng(10)

# ---------- Customers (60 rows + 1 duplicate) ----------
first = ['Ali', 'Sara', 'Omar', 'Hina', 'Bilal', 'Zara', 'Usman', 'Ayesha', 'Hamza', 'Fatima', 'Imran', 'Mariam']
last = ['Khan', 'Ahmed', 'Farooq', 'Malik', 'Raza', 'Iqbal', 'Noor', 'Butt']
customers = pd.DataFrame({
    'Customer_ID': [f'C{100 + i}' for i in range(60)],
    'Name': [f'{rng.choice(first)} {rng.choice(last)}' for _ in range(60)],
    'Gender': rng.choice(['Male', 'Female'], 60),
    'Age': rng.integers(18, 65, 60),
    'City': rng.choice(['Lahore', 'Karachi', 'Islamabad', 'Multan', 'Peshawar'], 60, p=[0.3, 0.3, 0.2, 0.1, 0.1]),
})
customers = pd.concat([customers, customers.iloc[[7]].assign(City='Karachi')], ignore_index=True)   # C107 twice (moved city)

# ---------- Products (32 rows) ----------
cat_price = {'Electronics': (5000, 150000), 'Fashion': (800, 9000), 'Home': (1500, 40000), 'Books': (500, 3000)}
categories = rng.choice(list(cat_price), 32, p=[0.3, 0.3, 0.2, 0.2])
products = pd.DataFrame({
    'Product_ID': [f'P{200 + i}' for i in range(32)],
    'Product': [f'{c[:4]}-Item-{i}' for i, c in enumerate(categories)],
    'Category': categories,
    'Price': [int(rng.integers(*cat_price[c]) // 10 * 10) for c in categories],
})

# ---------- Orders (160 rows, 2024-10 .. 2025-03) ----------
n_orders = 160
orders = pd.DataFrame({
    'Order_ID': np.arange(5000, 5000 + n_orders),
    'Customer_ID': rng.choice(customers['Customer_ID'].unique(), n_orders),
    'Product_ID': rng.choice(products['Product_ID'], n_orders),
    'Order_Date': pd.to_datetime('2024-10-01') + pd.to_timedelta(rng.integers(0, 182, n_orders), unit='D'),
    'Quantity': rng.integers(1, 5, n_orders),
})
orders.loc[[5, 40], 'Customer_ID'] = ['C999', 'C998']          # unknown customers
orders.loc[[12], 'Product_ID'] = ['P999']                        # unknown product
orders = orders.sort_values('Order_Date').reset_index(drop=True)

# ---------- Payments (from another system: key has a DIFFERENT NAME and TYPE) ----------
paid = orders.sample(150, random_state=3)
payments = pd.DataFrame({
    'order_ref': paid['Order_ID'].astype(str).values,          # text instead of int!
    'Payment_Method': rng.choice(['Card', 'Cash on Delivery', 'Wallet', 'Bank Transfer'], 150, p=[0.4, 0.35, 0.15, 0.1]),
    'Payment_Status': rng.choice(['Paid', 'Pending', 'Failed'], 150, p=[0.85, 0.1, 0.05]),
})
payments = pd.concat([payments,
                      pd.DataFrame({'order_ref': ['9001', '9002'], 'Payment_Method': ['Card', 'Wallet'], 'Payment_Status': ['Paid', 'Paid']}),
                      payments.iloc[[0]]], ignore_index=True)   # 2 payments for unknown orders + 1 duplicate payment

for name, t in [('customers', customers), ('products', products), ('orders', orders), ('payments', payments)]:
    print(f'{name:<10} {t.shape}')

## Step 1 — Data validation (types, missing values, key formats)

In [ ]:
def profile(name, df, key):
    print(f'{name:<10} rows={len(df):>3} | key={key} dtype={df[key].dtype} | key unique={df[key].is_unique} | '
          f'missing keys={df[key].isna().sum()} | missing cells={df.isna().sum().sum()}')

profile('customers', customers, 'Customer_ID')
profile('products', products, 'Product_ID')
profile('orders', orders, 'Order_ID')
profile('payments', payments, 'order_ref')
print('\norders.Order_ID dtype:', orders['Order_ID'].dtype, '| payments.order_ref dtype:', payments['order_ref'].dtype, '→ mismatch!')

## Step 2 — Duplicate-key checks

In [ ]:
dup_cust = customers[customers['Customer_ID'].duplicated(keep=False)]
print('Duplicated customers:'); print(dup_cust, '\n')
vc = payments['order_ref'].value_counts()
print('Orders with more than one payment record:', vc[vc > 1].to_dict())

# Fix: keep the LAST customer record (most recent info) and one payment per order
customers = customers.drop_duplicates(subset='Customer_ID', keep='last').reset_index(drop=True)
payments = payments.drop_duplicates().reset_index(drop=True)
payments['Order_ID'] = payments['order_ref'].astype(int)          # fix the dtype for merging
print('\nAfter fixing → customers unique:', customers['Customer_ID'].is_unique, '| payments unique:', payments['Order_ID'].is_unique)

## Step 3 — Inner join: orders that have both a known customer and a known product

In [ ]:
inner = (orders.merge(customers, on='Customer_ID', how='inner', validate='many_to_one')
               .merge(products, on='Product_ID', how='inner', validate='many_to_one'))
print(f'orders: {len(orders)} → inner join: {len(inner)} rows (orders with unknown customer/product dropped)')

## Step 4 — Left join: keep every order (main fact table), enrich with customers and products

In [ ]:
left = (orders.merge(customers, on='Customer_ID', how='left', validate='many_to_one')
              .merge(products, on='Product_ID', how='left', validate='many_to_one'))
print(f'Rows: {len(left)} (= orders) | orders missing customer info: {left["Name"].isna().sum()} | missing product info: {left["Product"].isna().sum()}')

## Step 5 — Outer join: reconcile customers and orders

In [ ]:
recon = pd.merge(customers[['Customer_ID', 'Name']], orders[['Order_ID', 'Customer_ID']], on='Customer_ID', how='outer', indicator=True)
print(recon['_merge'].value_counts())
print('Customers who never ordered:', recon.loc[recon['_merge'] == 'left_only', 'Customer_ID'].nunique())
print('Orders from unknown customers:', recon.loc[recon['_merge'] == 'right_only', 'Order_ID'].astype(int).tolist())

## Step 6 — Multiple-key join: city × category promotions

In [ ]:
promotions = pd.DataFrame({'City': ['Lahore', 'Lahore', 'Karachi', 'Islamabad', 'Multan'],
                           'Category': ['Electronics', 'Fashion', 'Electronics', 'Home', 'Books'],
                           'Promo_Discount': [0.10, 0.15, 0.05, 0.20, 0.25]})
left = left.merge(promotions, on=['City', 'Category'], how='left', validate='many_to_one')
left['Promo_Discount'] = left['Promo_Discount'].fillna(0)
print('Orders receiving a promotion:', (left['Promo_Discount'] > 0).sum())
print(left.loc[left['Promo_Discount'] > 0, ['Order_ID', 'City', 'Category', 'Promo_Discount']].head())

## Step 7 — Merge with different column names: payments (`order_ref`) — Step 8 — merge indicator — Step 9 — merge validation

In [ ]:
pay_check = pd.merge(orders[['Order_ID']], payments[['order_ref', 'Payment_Method', 'Payment_Status']].assign(order_ref=lambda d: d['order_ref'].astype(int)),
                     left_on='Order_ID', right_on='order_ref', how='outer', indicator=True, validate='one_to_one')
print(pay_check['_merge'].value_counts())
print('Orders without a payment record:', pay_check.loc[pay_check['_merge'] == 'left_only', 'Order_ID'].astype(int).tolist())
print('Payments for unknown orders    :', pay_check.loc[pay_check['_merge'] == 'right_only', 'order_ref'].astype(int).tolist())

left = left.merge(payments[['Order_ID', 'Payment_Method', 'Payment_Status']], on='Order_ID', how='left', validate='one_to_one')
left['Payment_Status'] = left['Payment_Status'].fillna('No Record')
print('\nRows still equal to orders:', len(left) == len(orders))

## Step 10 — Join using indexes: attach per-product sales totals

In [ ]:
prod_idx = products.set_index('Product_ID')
units_per_product = orders.groupby('Product_ID')['Quantity'].sum().rename('Units_Sold')
product_stats = prod_idx.join(units_per_product, how='left').fillna({'Units_Sold': 0})
product_stats['Units_Sold'] = product_stats['Units_Sold'].astype(int)
product_stats.sort_values('Units_Sold', ascending=False).head()

## Step 11 — Concatenate monthly datasets

In [ ]:
# Pretend each month arrived as a separate file
monthly_files = {str(p): g.drop(columns='Order_Month') for p, g in orders.assign(Order_Month=orders['Order_Date'].dt.to_period('M')).groupby('Order_Month')}
print({k: len(v) for k, v in monthly_files.items()})
recombined = pd.concat(monthly_files.values(), keys=monthly_files.keys(), names=['Month', 'Row']).reset_index(level='Month').reset_index(drop=True)
print('Recombined rows:', len(recombined), '| equals original:', len(recombined) == len(orders))
recombined.head()

## Step 12 — Combine all tables into one analytical DataFrame · Step 13 — Revenue

In [ ]:
analytics = left.copy()
analytics['Gross_Revenue'] = analytics['Quantity'] * analytics['Price']
analytics['Revenue'] = (analytics['Gross_Revenue'] * (1 - analytics['Promo_Discount'])).round(0)
analytics['Month'] = analytics['Order_Date'].dt.to_period('M')
cols = ['Order_ID', 'Order_Date', 'Month', 'Customer_ID', 'Name', 'Gender', 'Age', 'City', 'Product_ID', 'Product', 'Category',
        'Price', 'Quantity', 'Promo_Discount', 'Gross_Revenue', 'Revenue', 'Payment_Method', 'Payment_Status']
analytics = analytics[cols]
print(analytics.shape)
analytics.head()

## Step 14 — Analyse customers

In [ ]:
valid = analytics.dropna(subset=['Name', 'Product'])            # only fully matched orders for analysis
cust_stats = valid.groupby(['Customer_ID', 'Name']).agg(Orders=('Order_ID', 'count'), Revenue=('Revenue', 'sum')).reset_index()
all_customers = customers.merge(cust_stats[['Customer_ID', 'Orders', 'Revenue']], on='Customer_ID', how='left').fillna({'Orders': 0, 'Revenue': 0})
print('Customers without orders:', (all_customers['Orders'] == 0).sum())
print('Top 5 customers:'); print(all_customers.sort_values('Revenue', ascending=False).head()[['Customer_ID', 'Name', 'City', 'Orders', 'Revenue']])

## Step 15 — Analyse products · Step 16 — Analyse cities · Step 17 — Analyse payment methods

In [ ]:
print('Revenue by category:'); print(valid.groupby('Category')['Revenue'].sum().sort_values(ascending=False), '\n')
print('Top 5 products:'); print(valid.groupby(['Product_ID', 'Product'])['Revenue'].sum().nlargest(5), '\n')
print('City summary:')
print(valid.groupby('City').agg(Orders=('Order_ID', 'count'), Customers=('Customer_ID', 'nunique'), Revenue=('Revenue', 'sum'))
      .sort_values('Revenue', ascending=False), '\n')
print('Payment methods:')
print(pd.crosstab(valid['Payment_Method'].fillna('No Record'), valid['Payment_Status'], margins=True))

## Step 18 — Identify unmatched records · Step 19 — Detect data-quality problems

In [ ]:
quality = pd.Series({
    'Duplicate customer IDs found (fixed)': 1,
    'Duplicate payment rows found (fixed)': 1,
    'Orders with unknown customer': analytics['Name'].isna().sum(),
    'Orders with unknown product': analytics['Product'].isna().sum(),
    'Orders without payment record': (analytics['Payment_Status'] == 'No Record').sum(),
    'Payments for unknown orders': (pay_check['_merge'] == 'right_only').sum(),
    'Failed payments': (analytics['Payment_Status'] == 'Failed').sum(),
    'Customers who never ordered': (all_customers['Orders'] == 0).sum(),
    'Products never sold': (product_stats['Units_Sold'] == 0).sum(),
    'Key dtype mismatch (payments.order_ref text vs int)': 'fixed with astype(int)',
})
quality.to_frame('Count / status')

## Step 20 — Final analytical DataFrame

In [ ]:
final = analytics[analytics['Name'].notna() & analytics['Product'].notna()].reset_index(drop=True)
final['Paid'] = final['Payment_Status'] == 'Paid'
print(f'Final analytical table: {final.shape[0]} orders × {final.shape[1]} columns | no missing customer/product info')
print(f"Total revenue: {final['Revenue'].sum():,.0f} PKR | paid revenue: {final.loc[final['Paid'], 'Revenue'].sum():,.0f} PKR")
print('\nMonthly revenue:'); print(final.groupby('Month')['Revenue'].sum())
final.head()

## Why each combining method was selected

| Step | Method | Why this method |
|---|---|---|
| Enrich orders with customers/products | `merge(how='left', validate='many_to_one')` | Orders are the **fact table** (one row per event) — keep every order and **look up** details by key; validation guarantees the lookup can't multiply rows |
| Only complete orders (Step 3) | `merge(how='inner')` | When analysis requires both sides to exist |
| Reconciliation (Steps 5, 7–8) | `merge(how='outer', indicator=True)` | Shows what exists on **both** sides and what is missing on each |
| Promotions (Step 6) | `merge(on=['City', 'Category'])` | A promotion is identified by the **pair** (composite key) |
| Payments (Step 7) | `merge(left_on=..., right_on=...)` after `astype(int)` | Different key **name** and **type** in the payment system |
| Product stats (Step 10) | `DataFrame.join()` | Both sides were already **indexed** by Product_ID (a groupby result) |
| Monthly files (Step 11) | `pd.concat(..., keys=...)` | Same columns, different rows → **stacking**, not matching |
| Customer summary (Step 14) | aggregate → `merge` | Aggregating first keeps the merge one-to-one and avoids duplication |

---
<a id="sec-33"></a>
# Final Cheat Sheet

## `pd.concat()` — stack / glue
```python
pd.concat([df1, df2])                                   # stack rows (axis=0)
pd.concat([df1, df2], ignore_index=True)                # fresh 0..n-1 index
pd.concat([jan, feb], keys=['Jan', 'Feb'])              # remember the source (MultiIndex)
pd.concat([df1, df2], join='inner')                     # only shared columns
pd.concat([df1, df2], axis=1)                           # side by side, aligned on INDEX
pd.concat(list_of_dfs, ignore_index=True)               # many files: build a list, concat once
```

## `pd.merge()` — match on key columns
```python
pd.merge(left, right, on='key', how='inner')            # 'inner' | 'left' | 'right' | 'outer' | 'cross'
pd.merge(left, right, left_on='cust_id', right_on='id') # different key names
pd.merge(left, right, on=['k1', 'k2'])                  # composite key
pd.merge(left, right, on='key', suffixes=('_a', '_b'))  # overlapping column names
pd.merge(left, right, on='key', validate='many_to_one') # '1:1' | '1:m' | 'm:1' | 'm:m'
pd.merge(left, right, on='key', how='outer', indicator=True)   # _merge: both / left_only / right_only
pd.merge(left, right, left_on='key', right_index=True)  # column ↔ index
pd.merge_asof(a.sort_values('t'), b.sort_values('t'), on='t', direction='backward')   # nearest earlier match
pd.merge_ordered(a, b, on='date', fill_method='ffill')  # ordered merge with forward fill
```

| how | Keeps |
|---|---|
| `inner` | Keys in **both** |
| `left` | All **left** keys |
| `right` | All **right** keys |
| `outer` | **All** keys from both |

## `df.join()` — match on the index
```python
left.join(right)                          # index ↔ index, how='left'
left.join(right, how='inner'/'outer'/'right')
left.join(right, on='key')                # left column ↔ right index
left.join(right, lsuffix='_l', rsuffix='_r')
left.join([df2, df3])                     # several indexed tables
```

## Before every merge
```text
✔ key dtypes match          ✔ keys cleaned (strip/case/zeros)   ✔ lookup keys unique (is_unique)
✔ no missing keys           ✔ validate= the relationship         ✔ compare row counts before/after
✔ indicator=True to find unmatched rows                         ✔ check new NaNs after left joins
```

🎉 **You have completed Topic 10 — Combining DataFrames with Pandas!**